## Connecting LangGraph nodes to MCP server

We'll host the MCP server locally to demonstrate. To start the MCP server, run `python mcp_server_local.py` in the terminal to start (see `mcp_server_local.py` for how the tools are defined). This should start a server with:
- Name 'DemoServer'
- Has tools as defined in `mcp_server_local.py`
- Runs with a clean transport setup

LangGraph can discover and call those tools just like it would any other service.

In [1]:
# Start by loading environment variables (OpenAI model endpoint and key)
import os

from dotenv import load_dotenv
load_dotenv('.env')
OPENAI_API_ENDPOINT=os.getenv("OPENAI_API_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT_NAME=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME_52")
OPENAI_API_VERSION=os.getenv("OPENAI_API_VERSION")
RESOURCE_TENANT_ID=os.getenv("RESOURCE_TENANT_ID")

In [2]:
# Ensure all API Keys are removed from the environment variables.
# This may be necessary to prevent any default usage of API key variables by Langchain.
os.environ.pop("OPENAI_API_KEY", None)
os.environ.pop("AZURE_OPENAI_API_KEY", None)

In [3]:
from azure.identity import AzureCliCredential, get_bearer_token_provider

# Run 'azlogin' in terminal...
credential = AzureCliCredential(tenant_id=RESOURCE_TENANT_ID)
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

### Initiate LLM Resource

In [ ]:
# Import libraries:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import AzureChatOpenAI


[TensorFlow DLL Diagnostic] Analyzing: c:\anaconda3\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


Neither PyTorch nor TensorFlow >= 2.0 have been found.Models won't be available and only tokenizers, configurationand file/data utilities can be used.


In [5]:
# Define a state. Note the use of Annotated here in order to specify to langgraph how the message should be handled:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function in the annotation defines how this state key
    # should be updated (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

# Initiate the stategraph with the defined state:
graph_builder = StateGraph(State)

In [6]:
llm = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
    azure_endpoint=OPENAI_API_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version=OPENAI_API_VERSION,
)